In [0]:
print("Hello")

In [0]:
%sql SELECT CURRENT_TIMESTAMP()

In [0]:
%sql
create schema dev.aziz;

In [0]:
%sql
create table dev.aziz.demo (id int, name string )

![](/Volumes/dev/lekha/raw/Image/object-model.png)

In [0]:
(
    spark
    .read
    .csv("/Volumes/dev/naval/raw/csv/financial_dataset.csv",header=True,inferSchema=True)
    .write
    .mode("overwrite")
    .saveAsTable("dev.aziz.financial")
    )

In [0]:
df=spark.read.table("dev.aziz.financial")

In [0]:
df.filter("Branch='New York'").explain(True)

In [0]:
df.filter("Branch='New York'").display()

In [0]:
from pyspark.sql.functions import *
df.select(col("TransactionID").alias("transaction_id"))#.display()

In [0]:
df.withColumn("AmountUSD",round("AmountUSD")).withColumn("ingestion_date",current_date()).withColumn("env",lit("dev")).display()

In [0]:
  {
    "account_id": "ACC_00000001",
    "customer_id": "CUST_002184",
    "account_type": "Current",
    "account_number": "239618943251",
    "branch_code": "BR_009",
    "opening_date": "2019-10-16",
    "account_status": "Dormant",
    "current_balance": 77859.42,
    "available_balance": 138925.1,
    "interest_rate": 5.58,
    "minimum_balance": 1000,
    "overdraft_limit": 0,
    "last_transaction_date": "2023-12-21",
    "is_joint_account": true,
    "nominee_details": {
      "nominee_name": "Nominee791",
      "relationship": "Parent"
    },
    "created_timestamp": "2015-05-18T11:31:22Z"
  }

In [0]:
df = spark.read.json("/Volumes/dev/naval/raw/accounts.json")

In [0]:
df.display()

In [0]:
df = spark.read.option("multiLine", True).json("/Volumes/dev/naval/raw/accounts.json")


In [0]:
df = df.withColumn("ingestion_date", current_timestamp())
df = df.withColumnRenamed("account_id", "accountid")
df = df.withColumn("nominee_name", df["nominee_details.nominee_name"]) \
       .withColumn("relationship", df["nominee_details.relationship"])

In [0]:
df = df.drop("nominee_details")


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("dev.aziz.account_json_parsed")


In [0]:
%sql
create table dev.aziz.demo_ice (id int, name string ) using iceberg

In [0]:
test